# Phase 0 — Auto Ball Tracking Validation

Run order: top to bottom. Use **Runtime → Change runtime type → T4 GPU** before starting.

Outputs `tracks.json` + `preview_tv.mp4` + `preview_compare.mp4`. Watch the preview videos to decide go/no-go on the full pipeline.

## 1. Install dependencies

In [ ]:
!pip install -q ultralytics==8.3.0 opencv-python-headless==4.10.0.84 numpy scipy tqdm imageio[ffmpeg]
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Provide input video

Either upload a clip from your Insta360 X5 (stitched equirectangular MP4), or set `INPUT_URL` to a sample. Keep clips ≤ 60 sec for fast iteration.

In [ ]:
import os

# Option A: upload from local disk
USE_UPLOAD = True

# Option B: download from URL (e.g. a clip you put on R2)
INPUT_URL = ''  # e.g. 'https://pub-XXX.r2.dev/sample_x5_clip.mp4'

INPUT_PATH = '/content/input.mp4'

if USE_UPLOAD:
    from google.colab import files
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    os.rename(fname, INPUT_PATH)
elif INPUT_URL:
    !wget -q -O {INPUT_PATH} {INPUT_URL}

import cv2
cap = cv2.VideoCapture(INPUT_PATH)
FPS = cap.get(cv2.CAP_PROP_FPS)
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
N = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()
print(f'Input: {W}x{H} @ {FPS:.1f}fps, {N} frames, {N/FPS:.1f}s')
assert abs(W / H - 2.0) < 0.05, f'Expected equirectangular (2:1 aspect), got {W}x{H}'

## 3. Load YOLOv8 detector

Using `yolov8n.pt` (smallest, fastest). COCO-pretrained → classes 0=person, 32=sports ball. Good enough for validation; we fine-tune later if needed.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
model.to('cuda' if torch.cuda.is_available() else 'cpu')
print('Model loaded.')

## 4. Run detection per frame

**Strategy for equirectangular**: action stays near the equator (mid 33% vertically) on a level-tripod 360° rig. We crop that band before YOLO to (a) reduce distortion and (b) speed up inference 3x.

Coordinate output: `lon ∈ [-180, 180]`, `lat ∈ [-90, 90]`. Lon=0 is the direction the camera was facing at start.

In [ ]:
from tqdm import tqdm
import numpy as np

# Process at 10 Hz (every ~3 frames at 30fps) — matches output sample rate
SAMPLE_HZ = 10
stride = max(1, round(FPS / SAMPLE_HZ))

# Equator band crop (middle 33% vertically)
BAND_TOP = int(H * 0.33)
BAND_BOT = int(H * 0.67)
BAND_H = BAND_BOT - BAND_TOP

def px_to_lonlat(cx, cy):
    # cx in [0,W], cy in [BAND_TOP, BAND_BOT]
    lon = (cx / W) * 360.0 - 180.0
    lat = 90.0 - (cy / H) * 180.0
    return lon, lat

cap = cv2.VideoCapture(INPUT_PATH)
raw_detections = []  # list of dicts per processed frame

frame_idx = 0
pbar = tqdm(total=N, desc='Detecting')
while True:
    ret, frame = cap.read()
    if not ret: break
    if frame_idx % stride == 0:
        band = frame[BAND_TOP:BAND_BOT, :, :]
        # Resize for speed (keep aspect)
        det_w = 1280
        det_h = int(BAND_H * det_w / W)
        small = cv2.resize(band, (det_w, det_h))
        results = model.predict(small, conf=0.25, classes=[0, 32], verbose=False)
        players, ball = [], None
        if len(results) > 0:
            r = results[0]
            for box, cls, conf in zip(r.boxes.xyxy.cpu().numpy(),
                                       r.boxes.cls.cpu().numpy(),
                                       r.boxes.conf.cpu().numpy()):
                x1, y1, x2, y2 = box
                cx = (x1 + x2) / 2 * (W / det_w)
                cy_in_band = (y1 + y2) / 2 * (BAND_H / det_h)
                cy = BAND_TOP + cy_in_band
                lon, lat = px_to_lonlat(cx, cy)
                if int(cls) == 0:  # person
                    players.append({'lon': lon, 'lat': lat, 'conf': float(conf)})
                elif int(cls) == 32 and (ball is None or conf > ball['conf']):
                    ball = {'lon': lon, 'lat': lat, 'conf': float(conf)}
        raw_detections.append({
            't': frame_idx / FPS,
            'players': players,
            'ball': ball,
        })
    frame_idx += 1
    pbar.update(1)
pbar.close()
cap.release()

ball_hits = sum(1 for d in raw_detections if d['ball'] is not None)
print(f'Processed {len(raw_detections)} samples ({SAMPLE_HZ} Hz)')
print(f'Ball detected in {ball_hits}/{len(raw_detections)} = {ball_hits/len(raw_detections)*100:.1f}%')
print(f'Avg players detected: {np.mean([len(d["players"]) for d in raw_detections]):.1f}')

## 5. Compute action centroid + smooth

Fuse ball + player cluster into a single target point. Apply EMA + dead zone + lead. Output the camera state stream.

In [ ]:
EMA_ALPHA = 0.15           # lower = smoother, higher = more responsive
BALL_WEIGHT = 0.55         # weight of ball vs player centroid (when ball detected)
LEAD_FRAMES = 6            # how many samples ahead to lead the camera
DEAD_ZONE_DEG = 8.0        # don't move if target within this of current
FOV_TIGHT = 35.0
FOV_WIDE = 65.0

def angular_diff(a, b):
    d = (a - b + 180) % 360 - 180
    return d

def circular_mean(values):
    if not values: return None
    rads = np.deg2rad(values)
    s = np.mean(np.sin(rads)); c = np.mean(np.cos(rads))
    return float(np.rad2deg(np.arctan2(s, c)))

raw_targets = []
for d in raw_detections:
    players = d['players']
    ball = d['ball']
    if not players and not ball:
        raw_targets.append(None)
        continue
    player_lon = circular_mean([p['lon'] for p in players]) if players else None
    player_lat = float(np.mean([p['lat'] for p in players])) if players else None
    if ball and player_lon is not None:
        # weighted mix
        diff = angular_diff(ball['lon'], player_lon)
        tgt_lon = (player_lon + BALL_WEIGHT * diff) % 360
        if tgt_lon > 180: tgt_lon -= 360
        tgt_lat = (1 - BALL_WEIGHT) * player_lat + BALL_WEIGHT * ball['lat']
    elif ball:
        tgt_lon, tgt_lat = ball['lon'], ball['lat']
    else:
        tgt_lon, tgt_lat = player_lon, player_lat
    # FOV: tight if players spread is small, wide if spread is large
    if len(players) > 1:
        lons = np.array([p['lon'] for p in players])
        spread = np.std(lons)
        fov = float(np.clip(FOV_TIGHT + spread * 1.5, FOV_TIGHT, FOV_WIDE))
    else:
        fov = FOV_TIGHT + 10
    raw_targets.append({'t': d['t'], 'lon': tgt_lon, 'lat': tgt_lat, 'fov': fov, 'hasBall': ball is not None})

# Forward-fill gaps (lost detections) with last known
last = None
filled = []
for r in raw_targets:
    if r is None and last is not None:
        filled.append(dict(last))
    else:
        filled.append(r)
        if r is not None: last = r

# EMA smooth (circular for lon)
smoothed = []
sm_lon = sm_lat = sm_fov = None
for r in filled:
    if r is None: continue
    if sm_lon is None:
        sm_lon, sm_lat, sm_fov = r['lon'], r['lat'], r['fov']
    else:
        # Apply dead zone
        d_lon = angular_diff(r['lon'], sm_lon)
        d_lat = r['lat'] - sm_lat
        if abs(d_lon) > DEAD_ZONE_DEG or abs(d_lat) > DEAD_ZONE_DEG * 0.5:
            sm_lon = (sm_lon + EMA_ALPHA * d_lon) % 360
            if sm_lon > 180: sm_lon -= 360
            sm_lat = sm_lat + EMA_ALPHA * d_lat
        sm_fov = sm_fov + EMA_ALPHA * (r['fov'] - sm_fov)
    smoothed.append({'t': r['t'], 'lon': sm_lon, 'lat': sm_lat, 'fov': sm_fov})

# Apply lead (shift future N samples back)
led = []
for i, s in enumerate(smoothed):
    j = min(len(smoothed) - 1, i + LEAD_FRAMES)
    led.append({
        't': s['t'],
        'lon': smoothed[j]['lon'],
        'lat': smoothed[j]['lat'],
        'fov': smoothed[j]['fov'],
    })

print(f'Smoothed track: {len(led)} samples')

## 6. Write tracks.json (the contract for the client)

In [ ]:
import json

tracks = {
    'fps': SAMPLE_HZ,
    'duration': N / FPS,
    'trackingVersion': 1,
    'samples': [[round(s['t'], 3), round(s['lon'], 2), round(s['lat'], 2), round(s['fov'], 1)] for s in led]
}
with open('/content/tracks.json', 'w') as f:
    json.dump(tracks, f, separators=(',', ':'))
import os
size_kb = os.path.getsize('/content/tracks.json') / 1024
print(f'Wrote tracks.json: {size_kb:.1f} KB ({len(led)} samples)')
print('Per-minute size:', round(size_kb / (N/FPS/60), 1), 'KB/min')

## 7. Render TV-mode preview (THE VERDICT)

Re-projects the equirectangular source through a virtual camera following the smoothed track. Output: `preview_tv.mp4`.

**Watch this video. If it feels watchable → ship it. If not → tune params (EMA_ALPHA, DEAD_ZONE_DEG, LEAD_FRAMES) and re-run from step 5.**

In [ ]:
OUT_W, OUT_H = 1280, 720  # 16:9 TV crop

def render_perspective(equi, lon_deg, lat_deg, fov_deg, out_w, out_h):
    """Sample equirectangular `equi` through a perspective camera. Returns BGR image."""
    fh, fw = equi.shape[:2]
    # Build ray grid
    fov = np.deg2rad(fov_deg)
    f = (out_w / 2) / np.tan(fov / 2)
    cx, cy = out_w / 2, out_h / 2
    xs = np.arange(out_w) - cx
    ys = np.arange(out_h) - cy
    xx, yy = np.meshgrid(xs, ys)
    zs = np.full_like(xx, f, dtype=np.float32)
    dirs = np.stack([xx, yy, zs], axis=-1).astype(np.float32)
    norms = np.linalg.norm(dirs, axis=-1, keepdims=True)
    dirs /= norms
    # Rotate by lat (around X), then lon (around Y)
    lat = np.deg2rad(lat_deg)
    lon = np.deg2rad(lon_deg)
    cl, sl = np.cos(lat), np.sin(lat)
    co, so = np.cos(lon), np.sin(lon)
    Rx = np.array([[1,0,0],[0,cl,-sl],[0,sl,cl]], dtype=np.float32)
    Ry = np.array([[co,0,so],[0,1,0],[-so,0,co]], dtype=np.float32)
    R = Ry @ Rx
    rotated = dirs @ R.T
    x, y, z = rotated[..., 0], rotated[..., 1], rotated[..., 2]
    # Convert to equirectangular UVs
    theta = np.arctan2(x, z)
    phi = np.arcsin(np.clip(y, -1, 1))
    u = (theta / (2 * np.pi) + 0.5) * fw
    v = (phi / np.pi + 0.5) * fh
    return cv2.remap(equi, u.astype(np.float32), v.astype(np.float32),
                     interpolation=cv2.INTER_LINEAR, borderMode=cv2.BORDER_WRAP)

def lerp_track(t, samples):
    # Binary search
    import bisect
    times = [s[0] for s in samples]
    i = bisect.bisect_left(times, t)
    if i == 0: return samples[0][1], samples[0][2], samples[0][3]
    if i >= len(samples): return samples[-1][1], samples[-1][2], samples[-1][3]
    t0, lon0, lat0, fov0 = samples[i-1]
    t1, lon1, lat1, fov1 = samples[i]
    a = (t - t0) / max(1e-6, t1 - t0)
    # Circular lerp for lon
    dlon = ((lon1 - lon0 + 180) % 360) - 180
    lon = lon0 + a * dlon
    lat = lat0 + a * (lat1 - lat0)
    fov = fov0 + a * (fov1 - fov0)
    return lon, lat, fov

samples = tracks['samples']
cap = cv2.VideoCapture(INPUT_PATH)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out_tv = cv2.VideoWriter('/content/preview_tv.mp4', fourcc, FPS, (OUT_W, OUT_H))
out_cmp = cv2.VideoWriter('/content/preview_compare.mp4', fourcc, FPS, (OUT_W, OUT_H * 2))

frame_idx = 0
pbar = tqdm(total=N, desc='Rendering')
while True:
    ret, frame = cap.read()
    if not ret: break
    t = frame_idx / FPS
    lon, lat, fov = lerp_track(t, samples)
    tv = render_perspective(frame, lon, lat, fov, OUT_W, OUT_H)
    out_tv.write(tv)
    # Side-by-side: top = wide equirectangular thumbnail, bottom = TV crop
    wide_thumb = cv2.resize(frame, (OUT_W, OUT_H))
    # Draw indicator of where TV camera is pointing on the wide view
    cx_ind = int((lon + 180) / 360 * OUT_W)
    cy_ind = int((90 - lat) / 180 * OUT_H)
    cv2.rectangle(wide_thumb, (cx_ind-30, cy_ind-20), (cx_ind+30, cy_ind+20), (0, 255, 0), 2)
    stacked = np.vstack([wide_thumb, tv])
    out_cmp.write(stacked)
    frame_idx += 1
    pbar.update(1)
pbar.close()
cap.release()
out_tv.release()
out_cmp.release()
print('Done. Downloading...')

## 8. Download outputs

In [ ]:
from google.colab import files
files.download('/content/tracks.json')
files.download('/content/preview_tv.mp4')
files.download('/content/preview_compare.mp4')

## What to do with the results

1. Watch `preview_tv.mp4` end-to-end — does it feel like TV?
2. Watch `preview_compare.mp4` — does the green box on the wide view track sensibly?
3. Note any moments where:
   - Camera lags behind action → reduce `EMA_ALPHA` or increase `LEAD_FRAMES`
   - Camera jitters → increase `DEAD_ZONE_DEG`
   - Camera misses a goal or break → likely a detection gap; consider fine-tuning YOLO on your footage
4. Inspect `tracks.json` shape — confirm the schema is what the client needs:
   ```json
   {
     "fps": 10,
     "duration": 60.0,
     "trackingVersion": 1,
     "samples": [[t, lon, lat, fov], ...]
   }
   ```